In [2]:
import subprocess
subprocess.run(["pip", "install", "mlflow==2.13.0", "scikit-learn", "boto3"], capture_output=True)
print("Done")


Done


In [3]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import mlflow
import mlflow.sklearn
import os

# MLflow is pre-configured via MLFLOW_TRACKING_URI env var
mlflow.set_experiment("iris-classifier-notebook")

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Try changing these and re-running to see multiple runs in MLflow
params = {
    "n_estimators": 200,
    "max_depth": 10,
    "random_state": 42
}

with mlflow.start_run() as run:
    mlflow.log_params(params)
    
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average="weighted")
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    mlflow.sklearn.log_model(model, artifact_path="model")
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"MLflow tracking URI: {os.getenv('MLFLOW_TRACKING_URI')}")

# Register model
mlflow.register_model(f"runs:/{run.info.run_id}/model", "iris-classifier")
print("Model registered in MLflow registry")

   

Registered model 'iris-classifier' already exists. Creating a new version of this model...
2026/03/17 23:29:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 3


Accuracy: 1.0000
F1 Score: 1.0000
MLflow tracking URI: http://mlflow.default.svc.cluster.local:5000
Model registered in MLflow registry


Created version '3' of model 'iris-classifier'.
